# CLIP Fragment Matching — Turin Walk

This notebook matches visual fragments from the AI-generated image of Turin
to the 24 walk locations using CLIP.

**Inputs:**
- Fragment images: `fragments for clip/`
- Top TF-IDF words per location: `LOCATIONS - TOP WORDS FROM CONVO.xlsx`

**Output:**
- `FRAGMENTS TO LOCATION/fragment_to_location.xlsx`
- `FRAGMENTS TO LOCATION/fragment_to_location.csv`

In [1]:
!pip install torch torchvision pillow pandas open_clip_torch openpyxl -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
import os, ast
import torch
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter
import open_clip

FRAGMENTS_PATH  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\fragments for clip"
EXCEL_TOP_WORDS = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\VECTORIZING TEXT\EXCEL\LOCATIONS - TOP WORDS FROM CONVO.xlsx"
OUTPUT_FOLDER   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\IMAGE GENERATION - FRAGMENTS\FRAGMENTS TO LOCATION"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
OUTPUT_CSV   = os.path.join(OUTPUT_FOLDER, 'fragment_to_location.csv')
OUTPUT_EXCEL = os.path.join(OUTPUT_FOLDER, 'fragment_to_location.xlsx')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print('Setup complete.')

Device: cpu
Setup complete.


In [3]:
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
model     = model.to(device)
model.eval()
print('CLIP model loaded.')

C:\Users\elena\AppData\Local\Programs\Python\Python314\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP model loaded.


In [4]:
# Load fragments — exclude thumbnails
image_files = sorted([
    f for f in os.listdir(FRAGMENTS_PATH)
    if f.lower().endswith(('.png','.jpg','.jpeg','.webp'))
    and '_thumb' not in f.lower()
])
print(f'Fragments found: {len(image_files)}')
for f in image_files:
    print(f'  {f}')

Fragments found: 49
  Screenshot 2025-11-23 213815.png
  Screenshot 2025-11-23 213820.png
  Screenshot 2025-11-23 213826.png
  Screenshot 2025-11-23 213830.png
  Screenshot 2025-11-23 213833.png
  Screenshot 2025-11-23 213842.png
  Screenshot 2025-11-23 213845.png
  Screenshot 2025-11-23 213851.png
  Screenshot 2025-11-23 213855.png
  Screenshot 2025-11-23 213859.png
  Screenshot 2025-11-23 213904.png
  Screenshot 2025-11-23 213907.png
  Screenshot 2025-11-23 213910.png
  Screenshot 2025-11-23 213914.png
  Screenshot 2025-11-23 213917.png
  Screenshot 2025-11-23 213922.png
  Screenshot 2025-11-23 213926.png
  Screenshot 2025-11-23 213931.png
  Screenshot 2025-11-23 213936.png
  Screenshot 2025-11-23 213943.png
  Screenshot 2025-11-23 213953.png
  Screenshot 2025-11-23 213955.png
  Screenshot 2025-11-23 213959.png
  Screenshot 2025-11-23 214002.png
  Screenshot 2025-11-23 214009.png
  Screenshot 2025-11-23 214016.png
  Screenshot 2025-11-23 214021.png
  Screenshot 2025-11-23 214024.png


In [5]:
# Load top words — skip 2 garbage header rows
df_words = pd.read_excel(EXCEL_TOP_WORDS, header=None)
df_words = df_words.iloc[2:].reset_index(drop=True)
df_words.columns = ['location', 'top_words']

STOPWORDS = {
    'the','of','and','is','it','in','to','a','with','its',
    'that','this','as','are','was','for','on','at','be',
    'but','not','or','an','by','from','they','we','you'
}

locations_dict = {}
for _, row in df_words.iterrows():
    loc   = str(row['location']).strip()
    terms = str(row['top_words'])
    try:
        words = ast.literal_eval(terms)
    except:
        words = [w.strip().strip("'") for w in terms.strip('[]').split(',')]
    words = [w.strip().lower() for w in words
             if len(w.strip()) >= 3
             and w.strip().lower() not in STOPWORDS]
    locations_dict[loc] = words[:10]

print(f'Locations loaded: {len(locations_dict)}')
for loc, words in locations_dict.items():
    print(f'  {loc[:45]:45s} → {words[:4]}')

Locations loaded: 23
  Borgo Medievale (Parco del Valen              → ['it is', 'of the', 'children', 'whispers of']
  Café Paris                                    → ['coffee', 'café', 'café paris', 'paris']
  Castello del Valentino                        → ['the spirit', 'spirit', 'laughter and', 'could']
  Cattedrale di San Giovanni Batti              → ['silence', 'own', 'the sacred', 'sacred']
  Chiesa della Gran Madre di Dio (              → ['ascent', 'chiesa', 'chiesa della', 'della']
  Giardini Cavour                               → ['could', 'tap', 'vine', 'giardini']
  Il posto giusto del Baby Luna Pa              → ['bright', 'laughter', 'the trees', 'park']
  Monumental Arch to the Artillery              → ['arch', 'among', 'among the', 'for the']
  Museo Egizio (outside)                        → ['tired', 'now', 'stones', 'feels']
  Palazzo Carignano                             → ['care', 'brick', 'in their', 'palazzo']
  Palazzo Madama                                → [

In [6]:
# Encode all words with CLIP text encoder
loc_word_pairs = []
for loc, words in locations_dict.items():
    for w in words:
        loc_word_pairs.append((loc, w))

all_words = [pair[1] for pair in loc_word_pairs]
all_text_features = []

for i in range(0, len(all_words), 256):
    batch = all_words[i:i+256]
    with torch.no_grad():
        tokens = tokenizer(batch).to(device)
        feats  = model.encode_text(tokens)
        feats  = feats / feats.norm(dim=-1, keepdim=True)
        all_text_features.append(feats.cpu().numpy())

text_features = np.vstack(all_text_features)
print(f'Encoded {len(all_words)} words | shape: {text_features.shape}')

Encoded 143 words | shape: (143, 512)


In [7]:
# Match each fragment to its best location
results = []

for img_name in image_files:
    img_path = os.path.join(FRAGMENTS_PATH, img_name)
    try:
        image = preprocess(
            Image.open(img_path).convert('RGB')
        ).unsqueeze(0).to(device)
    except Exception as e:
        print(f'Error: {img_name} — {e}')
        continue

    with torch.no_grad():
        img_feat = model.encode_image(image)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        img_feat = img_feat.cpu().numpy()[0]

    sims      = text_features @ img_feat
    top5_idx  = np.argsort(-sims)[:5]
    top5_locs = [loc_word_pairs[i][0] for i in top5_idx]
    best_loc  = Counter(top5_locs).most_common(1)[0][0]
    best_sim  = round(float(sims[top5_idx[0]]), 4)
    top_words = ', '.join([loc_word_pairs[i][1] for i in top5_idx])

    results.append({
        'fragment':   img_name,
        'location':   best_loc,
        'similarity': best_sim,
        'top_words':  top_words,
    })
    print(f'  {img_name[:40]:40s} → {best_loc[:35]} ({best_sim})')

df_out = pd.DataFrame(results)
print(f'\nMatched {len(df_out)} fragments')

  Screenshot 2025-11-23 213815.png         → Via Giuseppe Garibaldi, 43–41 (0.2641)
  Screenshot 2025-11-23 213820.png         → Palazzo Carignano (0.2466)
  Screenshot 2025-11-23 213826.png         → Via Giuseppe Garibaldi, 43–41 (0.2601)
  Screenshot 2025-11-23 213830.png         → Via Giuseppe Garibaldi, 43–41 (0.2569)
  Screenshot 2025-11-23 213833.png         → Palazzo Carignano (0.2483)
  Screenshot 2025-11-23 213842.png         → Porta Palatina (0.2462)
  Screenshot 2025-11-23 213845.png         → Porta Palatina (0.257)
  Screenshot 2025-11-23 213851.png         → Piazza Carlo Alberto (0.2404)
  Screenshot 2025-11-23 213855.png         → Il posto giusto del Baby Luna Pa (0.2414)
  Screenshot 2025-11-23 213859.png         → Via Giuseppe Garibaldi, 43–41 (0.2369)
  Screenshot 2025-11-23 213904.png         → Via Giuseppe Garibaldi, 43–41 (0.2614)
  Screenshot 2025-11-23 213907.png         → Via Giuseppe Garibaldi, 43–41 (0.2554)
  Screenshot 2025-11-23 213910.png         → Porta Pa

In [8]:
# Save outputs
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
df_out.to_excel(OUTPUT_EXCEL, index=False)

print(f'CSV saved:   {OUTPUT_CSV}')
print(f'Excel saved: {OUTPUT_EXCEL}')
print()
print('Fragments per location:')
summary = df_out.groupby('location')['fragment'].count().sort_values(ascending=False)
for loc, count in summary.items():
    print(f'  {loc[:45]:45s} {count}')

CSV saved:   C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\IMAGE GENERATION - FRAGMENTS\FRAGMENTS TO LOCATION\fragment_to_location.csv
Excel saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\IMAGE GENERATION - FRAGMENTS\FRAGMENTS TO LOCATION\fragment_to_location.xlsx

Fragments per location:
  Porta Palatina                                25
  Via Giuseppe Garibaldi, 43–41                 12
  Palazzo Carignano                             5
  Il posto giusto del Baby Luna Pa              3
  Piazza Carlo Alberto                          1
  Piazza CLN                                    1
  Piazza Maria Teresa                           1
  Piazza Castello                               1
